### Practiced read_csv parameters

'''converters work with single value, not a df
.get func in dict retrieves the value of a specified key.
'''

''' The function now takes 'val' (a single string like 'F' or 'M')
def gender_full_form(val):
    lookup = {'F': 'Female', 'M': 'Male'}
    return lookup.get(val, val) # Returns the full form, or the original if not found

df_mat3 = pd.read_csv('student-mat.csv', sep=';', nrows=15, converters={'sex': gender_full_form})
df_mat3'''

cols_count = 33
df_mat = pd.read_csv('student-mat.csv', sep=';', names=rd.sample(st.ascii_letters, cols_count), header=None, ) 
df_mat = pd.read_csv('student-mat.csv', sep=';', header=0)
df_mat = pd.read_csv('student-mat.csv', sep=';', index_col='school')
df_mat = pd.read_csv('student-mat.csv', sep=';', skiprows=10, skipfooter=3)
df_mat = pd.read_csv('student-mat.csv', sep=';', nrows=15)
df_mat = pd.read_csv('student-mat.csv', sep=';', nrows=15)


df_mat = pd.read_csv('student-mat.csv', sep=';', header=0)
df_mat.isnull().sum().sum() # check null in entire df
to optimize the loading of the dataset
df = pd.read_csv(
    'student-mat.csv', 
    sep=';', 
    engine='pyarrow',
    usecols=['sex', 'age', 'G3'],
    dtype={'sex': 'category', 'age': 'int8', 'G3': 'int8'}
)


### Imported Libraries, Read File and Renamed Columns 

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_mat = pd.read_csv('student-mat.csv', sep=';')

df_mat.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [3]:
df_mat.rename(columns={'Pstatus':'Pcohstatus'}, inplace=True)
df_mat.columns

Index(['school', 'sex', 'age', 'address', 'famsize', 'Pcohstatus', 'Medu',
       'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='str')

In [4]:
df_mat.head()

,school,sex,age,address,famsize,Pcohstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


## Score Gap Analysis Between Groups
Count how many students fall at each G3 score level
Identify missing score levels — gaps where nobody scored
Find where student frequency drops sharply between consecutive score levels
Use that cliff point to split students into above-cliff and below-cliff groups
Compare the two groups on key attributes: studytime, failures, absences, Medu, Fedu, schoolsup
Check if G1 and G2 alone could have predicted which group a student ends up in — proving early intervention is possible

### Added student_id and changed its position

In [5]:
def add_unique_student_id(df, col_name, prefix):
    """
    Generates and adds a column of unique, padded student IDs to a DataFrame.

    This function calculates the number of rows in the DataFrame and creates 
    a sequence of IDs starting from 1, padded with leading zeros to a 
    width of 3 (e.g., 'std001', 'std002').

    Args:
        df (pd.DataFrame): The DataFrame to which the IDs will be added.
        col_name (str): The name of the new column to be created.
        prefix (str): The string prefix to go before the numeric ID (e.g., 'std').

    Returns:
        pd.DataFrame: The modified DataFrame containing the new ID column.
    """
    unique_id = [f"{prefix}{i:03d}" for i in range(1, len(df)+1)]
    df[col_name] = unique_id
    return df

df_mat = add_unique_student_id(df_mat, 'student_id', 'std') 
df_mat

,school,sex,age,address,famsize,Pcohstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,student_id
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,std001
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,std002
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,std003
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,std004
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,std005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,MS,M,20,U,LE3,A,2,2,services,services,...,5,4,4,5,4,11,9,9,9,std391
391,MS,M,17,U,LE3,T,3,1,services,services,...,4,5,3,4,2,3,14,16,16,std392
392,MS,M,21,R,GT3,T,1,1,other,other,...,5,3,3,3,3,3,10,8,7,std393
393,MS,M,18,R,LE3,T,3,2,services,other,...,4,1,3,4,5,0,11,12,10,std394


In [6]:
# 1. 'Pop' the column out (this removes it from its current spot and saves it)
col_to_move = df_mat.pop('student_id')

# 2. 'Insert' it at index 0
# Syntax: df.insert(location_index, column_name, value)
df_mat.insert(0, 'student_id', col_to_move)

In [7]:
df_mat.head()

,student_id,school,sex,age,address,famsize,Pcohstatus,Medu,Fedu,Mjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,std001,GP,F,18,U,GT3,A,4,4,at_home,...,4,3,4,1,1,3,6,5,6,6
1,std002,GP,F,17,U,GT3,T,1,1,at_home,...,5,3,3,1,1,3,4,5,5,6
2,std003,GP,F,15,U,LE3,T,1,1,at_home,...,4,3,2,2,3,3,10,7,8,10
3,std004,GP,F,15,U,GT3,T,4,2,health,...,3,2,2,1,1,5,2,15,14,15
4,std005,GP,F,16,U,GT3,T,3,3,other,...,4,3,2,1,2,5,4,6,10,10


### Made a separate df with id and G3 

In [8]:
stdt_G3_grades = df_mat[['student_id', 'G3']]
stdt_G3_grades

stdt_G3_grades_sorted = stdt_G3_grades.sort_values(by='G3', ascending=False, ignore_index=True)
stdt_G3_grades_sorted

,student_id,G3
0,std048,20
1,std287,19
2,std375,19
3,std111,19
4,std009,19
...,...,...
390,std342,0
391,std344,0
392,std368,0
393,std390,0


### Tried to use consecutive difference without dealing with duplicates

In [9]:
'''# Create a list to store results
differences = [0] # Start with 0 or None for the first row

for i in range(1, len(df)):
    current_score = df[col_name].iloc[i]
    previous_score = df[col_name].iloc[i-1]
    differences.append(current_score - previous_score)

df['score_gap'] = differences

def score_diff(df, col_name):
    # This creates a new column showing the change from the previous row
    df['diff'] = df[col_name].diff()
    return df

calc_diff_df = score_diff(stdt_G3_grades_sorted, 'G3')
calc_diff_df

stdt_G3_grades_sorted['diff'].unique()
stdt_G3_grades_sorted['diff'].nunique()

stdt_G3_grades_sorted = stdt_G3_grades_sorted.drop(columns=['diff'])
stdt_G3_grades_sorted = stdt_G3_grades_sorted.drop(columns=['score_gap'])
'''

"# Create a list to store results\ndifferences = [0] # Start with 0 or None for the first row\n\nfor i in range(1, len(df)):\n    current_score = df[col_name].iloc[i]\n    previous_score = df[col_name].iloc[i-1]\n    differences.append(current_score - previous_score)\n\ndf['score_gap'] = differences\n\ndef score_diff(df, col_name):\n    # This creates a new column showing the change from the previous row\n    df['diff'] = df[col_name].diff()\n    return df\n\ncalc_diff_df = score_diff(stdt_G3_grades_sorted, 'G3')\ncalc_diff_df\n\nstdt_G3_grades_sorted['diff'].unique()\nstdt_G3_grades_sorted['diff'].nunique()\n\nstdt_G3_grades_sorted = stdt_G3_grades_sorted.drop(columns=['diff'])\nstdt_G3_grades_sorted = stdt_G3_grades_sorted.drop(columns=['score_gap'])\n"

### At which G3 score level does student density collapse

*Step 1 — Count how many students got each G3 score*

In [10]:
score_counts = stdt_G3_grades_sorted['G3'].value_counts().sort_index(ascending=False)
print(score_counts)

G3
20     1
19     5
18    12
17     6
16    16
15    33
14    27
13    31
12    31
11    47
10    56
9     28
8     32
7      9
6     15
5      7
4      1
0     38
Name: count, dtype: int64


*Step 2 — Find missing scores*

In [11]:
all_scores = pd.Series(range(0, 21))
missing_scores = all_scores[~all_scores.isin(stdt_G3_grades_sorted['G3'])]
print(missing_scores)

1    1
2    2
3    3
dtype: int64


*# Step 3 — Calculate frequency drop between score levels*

In [12]:
score_counts_diff = score_counts.diff()
print(score_counts_diff)

G3
20     NaN
19     4.0
18     7.0
17    -6.0
16    10.0
15    17.0
14    -6.0
13     4.0
12     0.0
11    16.0
10     9.0
9    -28.0
8      4.0
7    -23.0
6      6.0
5     -8.0
4     -6.0
0     37.0
Name: count, dtype: float64


* The most crowded score is G3 = 10 with 56 students, followed by G3 = 11 with 47 students. This means the bulk of the class is sitting right at the passing boundary. That alone is a finding.
* G3 = 0 has 38 students. That is not a cliff — that is a wall. 38 students scored nothing. This is the most alarming number in your entire dataset.
* Cliff 1 — G3 = 10 to 9 is your main cliff. Student count drops by 28 the moment you cross below the passing grade. This is the natural academic divide in this class.
* Cliff 2 — G3 = 8 to 7 is a secondary cliff. Another 23-student drop. Students scoring 7 or below are a much smaller, more isolated group.
* Mild Anomaly = scored 15 and 6 more than 16 and 7 
* The Anomaly — G3 = 0 with 38 students is completely detached. These students likely did not sit the exam at all, or were disqualified. They need to be treated as a separate group in your analysis, not lumped with low scorers.

### Grouping of Student & Percentage of the Group 

In [13]:
tbl_data = {
    'Group': ['Strong', 'Passing', 'At-risk', 'Critical'],
    'G3 Range': ['15 - 20', '10 - 14', '7 - 9', '4 - 6']
}

# 1. Convert the dictionary to a DataFrame first
df = pd.DataFrame(tbl_data)

# 2. Define the logical order
group_order = ['Critical', 'At-risk', 'Passing', 'Strong']

# 3. Apply the categorical order to the DataFrame column
df['Group'] = pd.Categorical(df['Group'], categories=group_order, ordered=True)

# 4. Pivot the DataFrame
tble_grp_g3 = pd.pivot_table(df, index='Group', values='G3 Range', aggfunc='first')

print(tble_grp_g3)


         G3 Range
Group            
Critical    4 - 6
At-risk     7 - 9
Passing   10 - 14
Strong    15 - 20


In [14]:
stdt_G3_grades_sorted

,student_id,G3
0,std048,20
1,std287,19
2,std375,19
3,std111,19
4,std009,19
...,...,...
390,std342,0
391,std344,0
392,std368,0
393,std390,0


In [15]:
stdt_G3_grades_filtered = stdt_G3_grades_sorted[stdt_G3_grades_sorted['G3'] > 3]

stdt_G3_grades_filtered


,student_id,G3
0,std048,20
1,std287,19
2,std375,19
3,std111,19
4,std009,19
...,...,...
352,std374,5
353,std249,5
354,std080,5
355,std101,5


In [16]:
# 1. Define the score boundaries (the "bins")
# We need 5 numbers to create 4 gaps: [4-6], [7-9], [10-14], [15-20]
bins = [3, 6, 9, 14, 20] 

# 2. Define the corresponding labels
labels = ['Critical', 'At-risk', 'Passing', 'Strong']

# 3. Create the new column
stdt_G3_grades_filtered['performance_group'] = pd.cut(
    stdt_G3_grades_filtered['G3'], 
    bins=bins, 
    labels=labels
)

stdt_G3_grades_filtered

,student_id,G3,performance_group
0,std048,20,Strong
1,std287,19,Strong
2,std375,19,Strong
3,std111,19,Strong
4,std009,19,Strong
...,...,...,...
352,std374,5,Critical
353,std249,5,Critical
354,std080,5,Critical
355,std101,5,Critical


In [17]:
# 1. Get the counts for each group
group_counts = stdt_G3_grades_filtered['performance_group'].value_counts()

# 2. Calculate the percentage
# We use 357 as the divisor as requested
group_percentages = (group_counts / len(stdt_G3_grades_filtered)) * 100

# 3. Combine them into a nice summary table
summary_df = pd.DataFrame({
    'Count': group_counts,
    'Percentage (%)': group_percentages
})

print(summary_df)

                   Count  Percentage (%)
performance_group                       
Passing              192       53.781513
Strong                73       20.448179
At-risk               69       19.327731
Critical              23        6.442577


* Over half the class — 53.8% — is just barely passing. 
* Combined with At-risk and Critical, that means 25.8% of students are in danger, which is roughly 1 in every 4 students.
* The Strong group is only 20.4% — so genuinely high performers are actually the minority here.

### Comparing the two groups on attributes

In [18]:
# Main cliff is at G3 = 10 (passing threshold)
stdt_G3_grades_filtered['cliff_group'] = stdt_G3_grades_filtered['G3'].apply(
    lambda x: 'Above' if x >= 10 else 'Below'
)

In [19]:
stdt_G3_grades_filtered

,student_id,G3,performance_group,cliff_group
0,std048,20,Strong,Above
1,std287,19,Strong,Above
2,std375,19,Strong,Above
3,std111,19,Strong,Above
4,std009,19,Strong,Above
...,...,...,...,...
352,std374,5,Critical,Below
353,std249,5,Critical,Below
354,std080,5,Critical,Below
355,std101,5,Critical,Below


In [20]:
df_mat.columns

Index(['student_id', 'school', 'sex', 'age', 'address', 'famsize',
       'Pcohstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian',
       'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid',
       'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel',
       'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2',
       'G3'],
      dtype='str')

*Category & their Attributes*     
* Family Situation: famsize, Pcohstatus, Medu, Fedu, Mjob, Fjob, guardian, famsup, famrel, address
* Student Lifestyle & Wellbeing: sex, age, Dalc, Walc, health, romantic
* Study & School Related: school, studytime, failures, schoolsup, paid, absences, nursery, higher, reason, internet
* Outside Activities: traveltime, activities, freetime, goout



In [21]:
sdt_filtered_att = stdt_G3_grades_filtered.merge(
    df_mat[['student_id', 'studytime', 'failures', 'absences', 'schoolsup', 'Fedu', 'Medu']],
    on='student_id'
)

# Step 2 — verify
print(sdt_filtered_att.shape)
print(sdt_filtered_att.head())

(357, 10)
  student_id  G3 performance_group cliff_group  studytime  failures  absences  \
0     std048  20            Strong       Above          4         0         4   
1     std287  19            Strong       Above          3         0         5   
2     std375  19            Strong       Above          3         0         0   
3     std111  19            Strong       Above          1         0         6   
4     std009  19            Strong       Above          2         0         0   

  schoolsup  Fedu  Medu  
0        no     3     4  
1        no     2     2  
2        no     4     4  
3        no     4     4  
4        no     2     3  


In [22]:
sdt_filtered_att['failures'].value_counts() # mostly 0 
sdt_filtered_att['studytime'].value_counts() # mostly 2 to 5 hours 
sdt_filtered_att['schoolsup'].value_counts() # mostly no 
sdt_filtered_att['Medu'].value_counts() # mostly higher
sdt_filtered_att['Fedu'].value_counts() # mostly 5th to 9th grade
sdt_filtered_att.loc[273] # did highest 75 absences - at risk group


student_id            std277
G3                         9
performance_group    At-risk
cliff_group            Below
studytime                  2
failures                   0
absences                  75
schoolsup                 no
Fedu                       2
Medu                       3
Name: 273, dtype: object

In [23]:
'''sdt_filtered_att[['performance_group', 'studytime']].value_counts()
sdt_filtered_att[['performance_group', 'failures']].value_counts()
sdt_filtered_att[['performance_group', 'schoolsup']].value_counts()
sdt_filtered_att[['performance_group', 'Medu']].value_counts()
sdt_filtered_att[['performance_group', 'Fedu']].value_counts()
sdt_filtered_att['absences'].unique()
sdt_filtered_abs = sdt_filtered_att['absences'] == 0
sdt_filtered_abs.value_counts() # 77 stdn 0 absences '''

"sdt_filtered_att[['performance_group', 'studytime']].value_counts()\nsdt_filtered_att[['performance_group', 'failures']].value_counts()\nsdt_filtered_att[['performance_group', 'schoolsup']].value_counts()\nsdt_filtered_att[['performance_group', 'Medu']].value_counts()\nsdt_filtered_att[['performance_group', 'Fedu']].value_counts()\nsdt_filtered_att['absences'].unique()\nsdt_filtered_abs = sdt_filtered_att['absences'] == 0\nsdt_filtered_abs.value_counts() # 77 stdn 0 absences "

*For each ordinal attribute, find the most common category per cliff group:*

In [24]:
ordinal_cols = ['studytime', 'failures', 'Medu', 'Fedu']

cliff_mode = sdt_filtered_att.groupby('cliff_group')[ordinal_cols].agg(
    lambda x: x.mode()[0]
)
print(cliff_mode)

             studytime  failures  Medu  Fedu
cliff_group                                 
Above                2         0     4     2
Below                2         0     4     3


*for absences*

In [25]:
print(sdt_filtered_att.groupby('cliff_group')['absences'].median().round(2))

cliff_group
Above    4.0
Below    6.0
Name: absences, dtype: float64


* Studytime — Both groups most commonly study 2–5 hours. No difference. Study habits alone do not separate above from below cliff students in this dataset.
* Failures — Both groups mode is 0. No difference at the mode level. However remember from your earlier exploration, Below cliff had significantly more students with 1, 2, 3 failures. The mode hides that.
* Medu — Both groups mode is 4 (higher education). No difference.
* Fedu — Above cliff mode is 2 (5th–9th grade), Below cliff mode is 3 (secondary education). Surprisingly reversed — below cliff fathers are slightly more educated at the mode level. This is unexpected.
* Absences — Above cliff median is 4, Below cliff median is 6. Small but real difference. Below cliff students miss more school.

* The Problem- The mode is too blunt here. It flattens everything into one number and hides the real story, especially for failures which we know tells a different story when you look at the full distribution.

In [27]:
failures_dist = sdt_filtered_att.groupby(['cliff_group', 'failures']).size().unstack(fill_value=0)
print(failures_dist)

failures       0   1  2  3
cliff_group               
Above        234  24  3  4
Below         60  16  9  7


* 88% of Above cliff students have zero past failures vs only 65% of Below cliff students
* 35% of Below cliff students have at least one past failure vs only 12% of Above cliff students
* Past failures are 3x more common below the cliff

In [28]:
schoolsup_dist = sdt_filtered_att.groupby(['cliff_group', 'schoolsup']).size().unstack(fill_value=0)
print(schoolsup_dist)

schoolsup     no  yes
cliff_group          
Above        237   28
Below         70   22


* 24% of Below cliff students receive extra school support vs only 11% of Above cliff students.
* Below cliff students are twice as likely to have extra school support — but they are still below the cliff anyway. This means the support exists but is either not enough or coming too late.

Attributes and its Findings
* failures:    Below cliff has 3x more past failures
* absences:    Below cliff misses more school (4 vs 6 median)
* schoolsup:   Below cliff receives 2x more support but still underperforms

### Check if G1/G2 could predict group membership

In [29]:
sdt_filtered_att = sdt_filtered_att.merge(
    df_mat[['student_id', 'G1', 'G2']],
    on='student_id'
)
print(sdt_filtered_att.shape)

(357, 12)


*Median G1 and G2 Per Cliff Group*

In [30]:
early_warning = sdt_filtered_att.groupby('cliff_group')[['G1', 'G2']].median()
print(early_warning)

               G1    G2
cliff_group            
Above        12.0  12.0
Below         8.0   8.0


* Above cliff median is 12, Below cliff median is 8 — both periods. 
* Below cliff students were already 4 points behind from Period 1. 
* The gap never closed.

*Full Distribution of G1 and G2*

In [32]:
print(sdt_filtered_att.groupby('cliff_group')['G1'].value_counts().sort_index())

cliff_group  G1
Above        6      1
             7      6
             8     18
             9     13
             10    30
             11    35
             12    34
             13    33
             14    30
             15    24
             16    22
             17     8
             18     8
             19     3
Below        3      1
             5      5
             6     14
             7     20
             8     19
             9     14
             10    16
             11     3
Name: count, dtype: int64


In [33]:
print(sdt_filtered_att.groupby('cliff_group')['G2'].value_counts().sort_index())

cliff_group  G2
Above        8      6
             9     18
             10    39
             11    34
             12    41
             13    37
             14    23
             15    34
             16    13
             17     5
             18    12
             19     3
Below        5      9
             6     11
             7     17
             8     22
             9     27
             10     5
             11     1
Name: count, dtype: int64


* Above cliff G1 spreads from 6 to 19, centered around 10–13. Healthy spread with most students passing.
* Below cliff G1 spreads from 3 to 11, centered around 6–8. Most students were already failing in Period 1.
* Same pattern repeats in G2 — the groups never cross over.

*Check How Many Below Cliff Students Were Already Failing in G1*

In [34]:
sdt_filtered_att['G1_fail'] = sdt_filtered_att['G1'] < 10
sdt_filtered_att['G2_fail'] = sdt_filtered_att['G2'] < 10

early_fail = sdt_filtered_att.groupby('cliff_group')[['G1_fail', 'G2_fail']].mean() * 100
print(early_fail.round(2))

             G1_fail  G2_fail
cliff_group                  
Above          14.34     9.06
Below          79.35    93.48


* 79% of Below cliff students were already failing in G1. 
* By G2 that rose to 93%. 
* The school had clear warning signs from Period 1 and Period 2 before G3 even happened.

*Check Students Who Recovered vs Stayed Below*

In [35]:
sdt_filtered_att['consistent_fail'] = (
    (sdt_filtered_att['G1'] < 10) & 
    (sdt_filtered_att['G2'] < 10)
)

consistent = sdt_filtered_att.groupby('cliff_group')['consistent_fail'].mean() * 100
print(consistent.round(2))

cliff_group
Above     6.79
Below    76.09
Name: consistent_fail, dtype: float64


* 76% of Below cliff students were failing both G1 and G2. 
* Only 7% of Above cliff students were. 
* This is not bad luck in G3 — this is a pattern that existed from the very beginning.

## Findings

### 1. Were Below cliff students already struggling in G1 and G2?
Yes. 79% of Below cliff students were already failing in G1, rising to 93% by G2. The median score was 8 in both periods, compared to 12 for Above cliff students.

### 2. How early could the school have identified at-risk students?
As early as Period 1 (G1). 76% of Below cliff students were consistently failing both G1 and G2 before G3 was even recorded.

### 3. Does this prove early intervention is possible?
Yes. The warning signs were present from the very first grading period. A student failing G1 has a 79% chance of ending up below the cliff in G3. Early intervention after G1 results could have reached the majority of at-risk students.